In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

# Geospatial
import geopandas as gpd
from shapely.geometry import Point

# --------------------------------------------------
# Pfade
# --------------------------------------------------

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --------------------------------------------------
# Helper: Data Quality Report
# --------------------------------------------------

def data_quality_report(df, name):
    report = {
        "dataset": name,
        "rows": len(df),
        "missing_values": df.isna().sum().to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "columns": list(df.columns)
    }
    return report

# --------------------------------------------------
# Processing Pipeline
# --------------------------------------------------

reports = []

csv_files = glob.glob(os.path.join(RAW_DIR, "*.csv"))

if not csv_files:
    print("Keine CSV Dateien im raw Ordner gefunden.")

for file_path in csv_files:

    print(f"Verarbeite: {file_path}")

    df = pd.read_csv(file_path)

    # --------------------------------------------------
    # 1. Basic Cleaning
    # --------------------------------------------------

    df = df.drop_duplicates()

    # Pflichtspalten prüfen (FIRMS Standard)
    required_cols = ["latitude", "longitude"]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Fehlende Spalte: {col} in {file_path}")

    # Entferne ungültige Koordinaten
    df = df.dropna(subset=["latitude", "longitude"])
    df = df[(df["latitude"].between(-90, 90)) & (df["longitude"].between(-180, 180))]

    # --------------------------------------------------
    # 2. Georeferenzierung (Point Geometry)
    # --------------------------------------------------

    geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

    # --------------------------------------------------
    # 3. DBSCAN-ready Feature Engineering
    # --------------------------------------------------

    # Zeitfeatures (falls vorhanden)
    if "acq_date" in gdf.columns:
        gdf["acq_date"] = pd.to_datetime(gdf["acq_date"], errors="coerce")
        gdf["year"] = gdf["acq_date"].dt.year
        gdf["month"] = gdf["acq_date"].dt.month
        gdf["day"] = gdf["acq_date"].dt.day

    # numerische Features für Clustering
    cluster_df = gdf.copy()

    cluster_df["lat"] = cluster_df.geometry.y
    cluster_df["lon"] = cluster_df.geometry.x

    # optional FIRMS Features
    for col in ["brightness", "frp", "confidence"]:
        if col in cluster_df.columns:
            cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce")

    # final dataset für DBSCAN
    cluster_features = cluster_df[[c for c in ["lat", "lon", "brightness", "frp", "confidence"] if c in cluster_df.columns]]

        # --------------------------------------------------
    # 3b. Missing Values & Quality Handling
    # --------------------------------------------------

    na_replacements = {}

    # confidence NAs -> 0.5 setzen
    if "confidence" in cluster_df.columns:
        confidence_na_count = cluster_df["confidence"].isna().sum()
        cluster_df["confidence"] = cluster_df["confidence"].fillna(0.5)
        na_replacements["confidence_replaced"] = int(confidence_na_count)
    else:
        na_replacements["confidence_replaced"] = 0

    # missing values report
    report = data_quality_report(df, os.path.basename(file_path))
    report.update(na_replacements)

    # optional feature stats
    optional_stats = {}

    for col in ["brightness", "frp", "confidence"]:
        if col in cluster_df.columns:
            optional_stats[f"{col}_min"] = cluster_df[col].min(skipna=True)
            optional_stats[f"{col}_max"] = cluster_df[col].max(skipna=True)

    report.update(optional_stats)

    reports.append(report)

    # --------------------------------------------------
    # 4. Save outputs
    # --------------------------------------------------

    base_name = os.path.basename(file_path).replace(".csv", "")

    geo_path = os.path.join(PROCESSED_DIR, f"{base_name}_geo.geojson")
    cluster_path = os.path.join(PROCESSED_DIR, f"{base_name}_cluster.csv")

    gdf.to_file(geo_path, driver="GeoJSON")
    cluster_features.to_csv(cluster_path, index=False)

    print(f"Gespeichert GeoJSON: {geo_path}")
    print(f"Gespeichert Cluster CSV: {cluster_path}")

# --------------------------------------------------
# 5. Save Data Quality Report
# --------------------------------------------------

report_df = pd.DataFrame(reports)
report_path = os.path.join(PROCESSED_DIR, f"data_quality_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")

report_df.to_csv(report_path, index=False)

print("
Fertig.")
print(f"Report gespeichert: {report_path}")

# --------------------------------------------------
# 6. Zusammenfassung ausgeben
# --------------------------------------------------

print("
================ QUALITY SUMMARY ================")

for report in reports:

    print(f"
Dataset: {report['dataset']}")
    print(f"Rows: {report['rows']}")
    print(f"Duplicate Rows: {report['duplicate_rows']}")

    print("
Missing Values per Column:")
    for col, val in report["missing_values"].items():
        if val > 0:
            print(f"  - {col}: {val}")

    print("
NA Replacements:")
    print(f"  - confidence -> 0.5: {report.get('confidence_replaced', 0)}")

    print("
Feature Ranges:")

    for feature in ["brightness", "frp", "confidence"]:
        min_key = f"{feature}_min"
        max_key = f"{feature}_max"

        if min_key in report and max_key in report:
            print(f"  - {feature}: min={report[min_key]} | max={report[max_key]}")

print("
=================================================")
